# Localize Literals - Working with Datasets

This notebook shows how to:
1. Understand datasets (enums/literals)
2. Get localized text for dataset values
3. Use the UI text cache for automatic localization
4. Work with common datasets (Rarity, ItemNiche, Scope, etc.)

In [1]:
from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import AssetCache
from assetextractor.parsing.core.texts import StandardTextConverter

# Load assets
config = Config.from_json("config.json")
assets = AssetCache.load(config)
texts = assets.texts
ui_cache = assets.properties.ui_text_cache

# Set language
LANGUAGE = "english"
texts.converter = StandardTextConverter(LANGUAGE)

print("Assets loaded!")

Assets loaded!


## Understanding Datasets

Datasets are like enums - they define a set of valid string values (literals).

In [2]:
# Access the datasets cache
datasets = assets.datasets

print(f"Total datasets: {len(datasets.elements)}\n")

# Show first 20 dataset names
print("First 20 datasets:")
for i, dataset_name in enumerate(list(datasets.elements.keys())[:20], 1):
    dataset = datasets[dataset_name]
    print(f"{i:2d}. {dataset_name:30s} ({len(dataset.literals)} values)")

Total datasets: 534

First 20 datasets:
 1. AITargetType                   (6 values)
 2. AmbientZone                    (2 values)
 3. AttackCondition                (13 values)
 4. BuildingTerrainType            (8 values)
 5. BuildingType                   (8 values)
 6. ComparisonOperator             (5 values)
 7. CriticalErrorType              (5 values)
 8. DynamicVariationType           (3 values)
 9. EconomyIncomeCategory          (22 values)
10. EnvironmentEffectType          (8 values)
11. ErrorCategory                  (6 values)
12. FeedbackSequenceType           (9 values)
13. GameCameraControl              (2 values)
14. GameObjectState                (3 values)
15. HappinessCategory              (5 values)
16. HappinessState                 (5 values)
17. IncidentCommunication          (15 values)
18. IncidentGraphicEffect          (3 values)
19. IncidentType                   (6 values)
20. InputMode                      (2 values)


## Explore a Specific Dataset

Let's look at the `Rarity` dataset:

In [3]:
# Get the Rarity dataset
rarity_dataset = datasets["Rarity"]

print(f"Dataset: {rarity_dataset.name}")
print(f"Number of values: {len(rarity_dataset.literals)}\n")

print("Literal values:")
for literal in rarity_dataset.literals:
    print(f"  - {literal}")

Dataset: Rarity
Number of values: 8

Literal values:
  - Narrative
  - Common
  - Uncommon
  - Rare
  - Epic
  - Legendary
  - Quest
  - Unique


## Localize Dataset Literals (Manual Method)

Without the UI text cache, you'd need to manually look up text IDs:

In [4]:
# Get an item's rarity attribute
item = assets.get(80510)
rarity_attr = item.Item.Rarity

# The raw value
rarity_literal = rarity_attr()
print(f"Rarity literal: {rarity_literal}")

# To get localized text, we'd need to:
# 1. Know the text ID for "Legendary"
# 2. Look it up in the texts cache
# This is tedious and requires manual mapping!

Rarity literal: Legendary


## Localize Using UI Text Cache (Easy Method)

The UI text cache automatically maps literals to localized text:

In [5]:
# Get an item's rarity with localization
item = assets.get(80510)
rarity_attr = item.Item.Rarity

# Method 1: Use ui_text property
if hasattr(rarity_attr, 'ui_text') and rarity_attr.ui_text:
    localized_rarity = rarity_attr.ui_text.values.get(LANGUAGE, 'N/A')
    print(f"Rarity (localized): {localized_rarity}")

# Method 2: Direct cache lookup
rarity_literal = rarity_attr()
mapping = ui_cache.get_ui_text("Rarity", rarity_literal)
if mapping:
    text_obj = texts.elements.get(mapping.text_id)
    if text_obj:
        localized = text_obj.values.get(LANGUAGE)
        print(f"Rarity (via cache): {localized}")

Rarity (localized): Legendary
Rarity (via cache): Legendary


## Localize All Values in a Dataset

Create a complete mapping of all rarity values:

In [6]:
def localize_dataset(dataset_name, language=LANGUAGE):
    """Create a mapping of all literals to localized text."""
    dataset = datasets.get(dataset_name)
    if not dataset:
        return {}
    
    result = {}
    
    for literal in dataset.literals:
        # Try to get UI text mapping
        mapping = ui_cache.get_ui_text(dataset_name, literal)
        if mapping:
            text_obj = texts.elements.get(mapping.text_id)
            if text_obj:
                localized = text_obj.values.get(language, literal)
                result[literal] = localized
            else:
                result[literal] = literal
        else:
            result[literal] = literal  # No mapping, use literal
    
    return result

# Localize all rarity values
rarity_localized = localize_dataset("Rarity")

print("Rarity Dataset (localized):\n")
for literal, localized_text in rarity_localized.items():
    print(f"  {literal:20s} → {localized_text}")

Rarity Dataset (localized):

  Narrative            → Special Item
  Common               → Common
  Uncommon             → Uncommon
  Rare                 → Rare
  Epic                 → Epic
  Legendary            → Legendary
  Quest                → Quest Item
  Unique               → Unique


## Common Datasets

Let's localize several commonly used datasets:

In [7]:
# Common datasets
common_datasets = [
    "Rarity",
    "ItemNiche",
    "ItemAllocation",
    "Scope",
]

for dataset_name in common_datasets:
    if dataset_name not in datasets.elements:
        print(f"\n{dataset_name}: Not found")
        continue
    
    print(f"\n{'='*60}")
    print(f"{dataset_name}")
    print('='*60)
    
    localized = localize_dataset(dataset_name)
    
    for literal, text in list(localized.items())[:10]:  # Show first 10
        print(f"  {literal:25s} → {text}")
    
    if len(localized) > 10:
        print(f"  ... and {len(localized) - 10} more")


Rarity
  Narrative                 → Special Item
  Common                    → Common
  Uncommon                  → Uncommon
  Rare                      → Rare
  Epic                      → Epic
  Legendary                 → Legendary
  Quest                     → Quest Item
  Unique                    → Unique

ItemNiche
  None                      → None
  Finance                   → Finance
  Religion                  → Religion
  Research                  → Research
  Culture                   → Culture
  Economy                   → Economy
  Agriculture               → Nature
  Diplomacy                 → Civic
  Military                  → Military
  Nautics                   → Seafaring

ItemAllocation
  None                      → Item
  Ship                      → Captain
  Villa                     → Specialist

Scope
  Local                     → Governor's Villa:
  ModuleOwner               → Main Building:
  StreetDistance            → connected:
  Radius                

## Item Properties with Localized Literals

Extract complete item information with all literals localized:

In [8]:
def extract_item_with_localized_literals(item_guid, language=LANGUAGE):
    """Extract item data with all literals localized."""
    item = assets.get(item_guid)
    if not item:
        return None
    
    result = {
        'guid': item.guid,
        'name': item.text.values.get(language, 'N/A') if item.text else 'N/A',
    }
    
    # Rarity
    if hasattr(item.Item, 'Rarity'):
        rarity_attr = item.Item.Rarity
        result['rarity_literal'] = rarity_attr()
        
        if hasattr(rarity_attr, 'ui_text') and rarity_attr.ui_text:
            result['rarity_text'] = rarity_attr.ui_text.values.get(language, 'N/A')
    
    # Niche
    if hasattr(item.Item, 'Niche'):
        niche_attr = item.Item.Niche
        result['niche_literal'] = niche_attr()
        
        if hasattr(niche_attr, 'ui_text') and niche_attr.ui_text:
            result['niche_text'] = niche_attr.ui_text.values.get(language, 'N/A')
    
    # Allocation
    if hasattr(item.Item, 'Allocation'):
        alloc_attr = item.Item.Allocation
        result['allocation_literal'] = alloc_attr()
        
        if hasattr(alloc_attr, 'ui_text') and alloc_attr.ui_text:
            result['allocation_text'] = alloc_attr.ui_text.values.get(language, 'N/A')
    
    # Scope
    if hasattr(item.Effect, 'EffectScope'):
        scope_attr = item.Effect.EffectScope
        result['scope_literal'] = scope_attr()
        
        if hasattr(scope_attr, 'ui_text') and scope_attr.ui_text:
            result['scope_text'] = scope_attr.ui_text.values.get(language, 'N/A')
    
    return result

# Extract item with localized literals
item_data = extract_item_with_localized_literals(80510)

if item_data:
    print("Item with Localized Literals:\n")
    print(f"Name: {item_data['name']}")
    print(f"GUID: {item_data['guid']}\n")
    
    if 'rarity_literal' in item_data:
        print(f"Rarity:")
        print(f"  Literal: {item_data['rarity_literal']}")
        print(f"  Text: {item_data.get('rarity_text', 'N/A')}\n")
    
    if 'niche_literal' in item_data:
        print(f"Niche:")
        print(f"  Literal: {item_data['niche_literal']}")
        print(f"  Text: {item_data.get('niche_text', 'N/A')}\n")
    
    if 'allocation_literal' in item_data:
        print(f"Allocation:")
        print(f"  Literal: {item_data['allocation_literal']}")
        print(f"  Text: {item_data.get('allocation_text', 'N/A')}\n")
    
    if 'scope_literal' in item_data:
        print(f"Scope:")
        print(f"  Literal: {item_data['scope_literal']}")
        print(f"  Text: {item_data.get('scope_text', 'N/A')}")

Item with Localized Literals:

Name: Zorsines, Sarmatian Swordshaper
GUID: 80510

Rarity:
  Literal: Legendary
  Text: Legendary

Niche:
  Literal: Economy
  Text: Economy

Allocation:
  Literal: Villa
  Text: Specialist

Scope:
  Literal: Radius
  Text: in range:


## Export Localized Dataset Dictionary

Export all dataset localizations for use in other applications:

In [9]:
import json
from pathlib import Path

# Create a comprehensive dataset localization dictionary
dataset_localizations = {}

important_datasets = [
    "Rarity",
    "ItemNiche",
    "ItemAllocation",
    "Scope",
    "ItemType",
    "BuffCategoryType",
]

for dataset_name in important_datasets:
    if dataset_name in datasets.elements:
        dataset_localizations[dataset_name] = localize_dataset(dataset_name)

# Save to JSON
output_dir = Path("results/example")
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "dataset_localizations.json"

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(dataset_localizations, f, indent=2, ensure_ascii=False)

print(f"Exported {len(dataset_localizations)} datasets to: {output_file}")
print(f"\nTotal literals localized: {sum(len(v) for v in dataset_localizations.values())}")

Exported 5 datasets to: results\example\dataset_localizations.json

Total literals localized: 37


## Check Dataset Coverage

See which datasets have UI text mappings available:

In [10]:
# Check which datasets are supported by UI cache
print("Dataset Coverage in UI Text Cache:\n")

covered_datasets = set()
for dataset_name in list(datasets.elements.keys())[:30]:  # Check first 30
    dataset = datasets[dataset_name]
    
    # Check if any literal has a mapping
    has_mapping = False
    for literal in dataset.literals[:5]:  # Check first 5 literals
        if ui_cache.get_ui_text(dataset_name, literal):
            has_mapping = True
            break
    
    if has_mapping:
        covered_datasets.add(dataset_name)
        status = "✓ Covered"
    else:
        status = "✗ Not covered"
    
    print(f"  {dataset_name:30s} {status}")

print(f"\nCovered: {len(covered_datasets)} datasets")

Dataset Coverage in UI Text Cache:

  AITargetType                   ✗ Not covered
  AmbientZone                    ✗ Not covered
  AttackCondition                ✗ Not covered
  BuildingTerrainType            ✗ Not covered
  BuildingType                   ✗ Not covered
  ComparisonOperator             ✗ Not covered
  CriticalErrorType              ✗ Not covered
  DynamicVariationType           ✗ Not covered
  EconomyIncomeCategory          ✗ Not covered
  EnvironmentEffectType          ✗ Not covered
  ErrorCategory                  ✗ Not covered
  FeedbackSequenceType           ✗ Not covered
  GameCameraControl              ✗ Not covered
  GameObjectState                ✗ Not covered
  HappinessCategory              ✗ Not covered
  HappinessState                 ✗ Not covered
  IncidentCommunication          ✗ Not covered
  IncidentGraphicEffect          ✗ Not covered
  IncidentType                   ✓ Covered
  InputMode                      ✗ Not covered
  LockScope                 

## Summary

Key takeaways:
1. Datasets define valid literal values (like enums)
2. Use `.ui_text` property on attributes for automatic localization
3. Use `ui_cache.get_ui_text(dataset, literal)` for direct lookups
4. Not all datasets have UI text mappings - some are internal only

## Next Steps

- `02_localization.ipynb` - More about working with texts
- `04_buffs_and_effects.ipynb` - See localized buff type names